In [ ]:
!pip install -q transformers sentence-transformers fuzzywuzzy[speedup] xgboost lightgbm

import pandas as pd
import numpy as np
import re, html
from tqdm import tqdm

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv('questions.csv')
df = df.dropna(subset=['question1','question2','is_duplicate'])
df.head()


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

CONTRACTIONS = {
    "can't":"can not","won't":"will not","don't":"do not","i'm":"i am",
    "it's":"it is","you're":"you are","they're":"they are","that's":"that is"

}

def decontract(text):
    text = text.lower()
    for c, full in CONTRACTIONS.items():
        text = re.sub(r"\b" + re.escape(c) + r"\b", full, text)
    return text

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'<.*?>', ' ', text)        # removeing HTML tags
    text = decontract(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # keep alnum + space
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

for col in ['question1','question2']:
    df[col + '_clean'] = df[col].astype(str).apply(clean_text)

df[['question1_clean','question2_clean']].head()


In [ ]:
def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def longest_common_substring(a, b):
    # dynamic programming LCS (substring)
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    longest = 0
    for i in range(m):
        for j in range(n):
            if a[i] == b[j]:
                dp[i+1][j+1] = dp[i][j] + 1
                longest = max(longest, dp[i+1][j+1])
    return longest

def length_features(q1, q2):
    l1, l2 = len(q1), len(q2)
    abs_diff = abs(l1 - l2)
    mean_len = (l1 + l2) / 2
    lcs = longest_common_substring(q1, q2)
    lcs_ratio = lcs / (min(l1, l2) + 1e-6)
    return abs_diff, mean_len, lcs_ratio

from fuzzywuzzy import fuzz

def build_basic_features(df):
    feats = {}
    feats['jaccard'] = df.apply(lambda r: jaccard(r['question1_clean'],
                                                  r['question2_clean']), axis=1)
    feats['len_diff'], feats['mean_len'], feats['lcs_ratio'] = zip(*df.apply(
        lambda r: length_features(r['question1_clean'], r['question2_clean']), axis=1))
    feats['fuzz_ratio'] = df.apply(lambda r: fuzz.QRatio(r['question1_clean'],
                                                         r['question2_clean']), axis=1)
    feats['fuzz_partial'] = df.apply(lambda r: fuzz.partial_ratio(r['question1_clean'],
                                                                  r['question2_clean']), axis=1)
    feats['fuzz_token_sort'] = df.apply(lambda r: fuzz.token_sort_ratio(r['question1_clean'],
                                                                        r['question2_clean']), axis=1)
    feats['fuzz_token_set'] = df.apply(lambda r: fuzz.token_set_ratio(r['question1_clean'],
                                                                      r['question2_clean']), axis=1)
    return pd.DataFrame(feats)

basic_feats = build_basic_features(df)
basic_feats.head()


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

def bert_cosine_batch(q1_list, q2_list, batch_size=256):
    sims = []
    for i in range(0, len(q1_list), batch_size):
        q1_batch = q1_list[i:i+batch_size].tolist()
        q2_batch = q2_list[i:i+batch_size].tolist()
        e1 = model.encode(q1_batch, convert_to_numpy=True, show_progress_bar=False)
        e2 = model.encode(q2_batch, convert_to_numpy=True, show_progress_bar=False)
        s = np.diag(cosine_similarity(e1, e2))
        sims.extend(s)
    return np.array(sims)

df['bert_cosine'] = bert_cosine_batch(df['question1_clean'], df['question2_clean'])
df['bert_cosine'].head()


In [ ]:
X = pd.concat([basic_feats, df[['bert_cosine']]], axis=1)
y = df['is_duplicate'].astype(int)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb


In [ ]:
def eval_model(clf, name):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
    print(f'{name}: acc={acc:.3f}, precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}')


In [ ]:
lr = LogisticRegression(max_iter=200, n_jobs=-1)
rf = RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42)
gb = GradientBoostingClassifier(random_state=42)
xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, objective='binary:logistic',
    tree_method='hist', eval_metric='logloss'
)
lgb_clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, objective='binary'
)

for clf, name in [
    (lr, 'LogisticRegression'),
    (rf, 'RandomForest'),
    (gb, 'GradientBoosting'),
    (xgb_clf, 'XGBoost'),
    (lgb_clf, 'LightGBM')
]:
    eval_model(clf, name)


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


In [ ]:
def pos_tags(text):
    tokens = text.split()
    return nltk.pos_tag(tokens)

def pos_overlap_counts(q1, q2):
    tags1 = pos_tags(q1)
    tags2 = pos_tags(q2)

    nouns1 = {w for w, t in tags1 if t.startswith('NN')}
    nouns2 = {w for w, t in tags2 if t.startswith('NN')}
    verbs1 = {w for w, t in tags1 if t.startswith('VB')}
    verbs2 = {w for w, t in tags2 if t.startswith('VB')}
    adjs1 = {w for w, t in tags1 if t.startswith('JJ')}
    adjs2 = {w for w, t in tags2 if t.startswith('JJ')}

    noun_overlap = len(nouns1 & nouns2)
    verb_overlap = len(verbs1 & verbs2)
    adj_overlap = len(adjs1 & adjs2)

    return noun_overlap, verb_overlap, adj_overlap, len(nouns1), len(nouns2), len(verbs1), len(verbs2)

pos_feats = df.apply(
    lambda r: pos_overlap_counts(r['question1_clean'], r['question2_clean']),
    axis=1, result_type='expand'
)


In [ ]:
df['question1_clean'] = df['question1_clean'].astype(str).fillna('')
df['question2_clean'] = df['question2_clean'].astype(str).fillna('')


In [ ]:
from rake_nltk import Rake
from nltk.corpus import stopwords
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
rake_extractor = Rake(stopwords=stop_words)

# Ensuring clean columns are strings with no NaN
df['question1_clean'] = df['question1_clean'].astype(str).fillna('')
df['question2_clean'] = df['question2_clean'].astype(str).fillna('')

def rake_keywords(text, max_kw=10):
    # force to string
    if text is None or (isinstance(text, float) and np.isnan(text)):
        text = ''
    text = str(text)
    if not text.strip():
        return set()
    rake_extractor.extract_keywords_from_text(text)
    phrases = rake_extractor.get_ranked_phrases()

    words = set()
    for kw in phrases[:max_kw]:
        kw = str(kw)
        if not kw:
            continue
        for w in kw.split():
            words.add(w)
    return words

def rake_overlap_stats(q1, q2):
    k1 = rake_keywords(q1)
    k2 = rake_keywords(q2)
    inter = k1 & k2
    union = k1 | k2
    overlap_count = len(inter)
    jaccard = overlap_count / (len(union) + 1e-6)
    return overlap_count, jaccard, len(k1), len(k2)

rake_feats = df.apply(
    lambda r: rake_overlap_stats(r['question1_clean'], r['question2_clean']),
    axis=1, result_type='expand'
)
rake_feats.columns = [
    'rake_overlap_count', 'rake_jaccard', 'rake_len_q1', 'rake_len_q2'
]


In [ ]:
# X:previous features (basic + bert_cosine + any extras)
# pos_feats, rake_feats: just computed

X_full = pd.concat(
    [X.reset_index(drop=True),
     pos_feats.reset_index(drop=True),
     rake_feats.reset_index(drop=True)],
    axis=1
)
X_full = pd.concat(
    [X.reset_index(drop=True),
     pos_feats.reset_index(drop=True),
     rake_feats.reset_index(drop=True)],
    axis=1
)

# Ensuring all feature names are strings
X_full.columns = X_full.columns.astype(str)

y = df['is_duplicate'].astype(int)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

def eval_model(clf, name):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
    print(f'{name}: acc={acc:.3f}, precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}')

rf_pos_rake = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

lgb_pos_rake = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary'
)

eval_model(rf_pos_rake, 'RandomForest_POS_RAKE')
eval_model(lgb_pos_rake, 'LightGBM_POS_RAKE')


In [ ]:
# Make sure feature names are strings
X_full.columns = X_full.columns.astype(str)

def build_features_for_pair_full(q1, q2):
    # Base DataFrame
    tmp = pd.DataFrame({'question1': [q1], 'question2': [q2]})

    # Same preprocessing as training
    tmp['question1_clean'] = tmp['question1'].astype(str).apply(clean_text)
    tmp['question2_clean'] = tmp['question2'].astype(str).apply(clean_text)

    # 1) Base similarity/length/fuzzy features
    basic = build_basic_features(tmp)

    # 2) BERT cosine similarity
    tmp['bert_cosine'] = bert_cosine_batch(
        tmp['question1_clean'], tmp['question2_clean'], batch_size=1
    )
    X_base = pd.concat([basic, tmp[['bert_cosine']]], axis=1)

    # 3) POS features for this pair
    pos_single = pd.DataFrame(
        [pos_overlap_counts(tmp['question1_clean'][0], tmp['question2_clean'][0])],
        columns=pos_feats.columns
    )

    # 4) RAKE features for this pair
    rake_single = pd.DataFrame(
        [rake_overlap_stats(tmp['question1_clean'][0], tmp['question2_clean'][0])],
        columns=rake_feats.columns
    )

    # Concatenate
    X_pair = pd.concat([X_base, pos_single, rake_single], axis=1)
    X_pair.columns = X_pair.columns.astype(str)

    # Reorder/align columns exactly as in training
    X_pair = X_pair.reindex(columns=X_full.columns, fill_value=0)

    return X_pair

def predict_duplicate_full(q1, q2, clf):
    X_pair = build_features_for_pair_full(q1, q2)
    prob = clf.predict_proba(X_pair)[0, 1]
    label = int(prob >= 0.5)
    return label, prob


In [ ]:
q1 = "What are the opinions of Modi about Pamela Anderson?"
q2 = "Why do Episcopalians pray the rosary, and what does it mean to do so?"

label, prob = predict_duplicate_full(q1, q2, lgb_pos_rake)  # or rf_pos_rake
print("Duplicate?", bool(label), "Probability:", prob)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans = KMeans(n_clusters=2, random_state=42)
clabels = kmeans.fit_predict(X_full)
print("Silhouette:", silhouette_score(X_full, clabels))


In [ ]:
from sklearn.manifold import TSNE
import numpy as np, matplotlib.pyplot as plt

idx = np.random.choice(len(X_full), size=10000, replace=False)
X_sample = X_full.iloc[idx]
y_sample = y.iloc[idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_2d = tsne.fit_transform(X_sample)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=y_sample, cmap='coolwarm', s=4, alpha=0.6)
plt.title('t-SNE of Question Pair Features')
plt.show()


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_NUM_WORDS = 50000
MAX_SEQ_LEN = 30

texts = pd.concat([df['question1_clean'], df['question2_clean']]).astype(str)

tokenizer = Tokenizer(num_words=MAX_NUM_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

seq1 = tokenizer.texts_to_sequences(df['question1_clean'].astype(str))
seq2 = tokenizer.texts_to_sequences(df['question2_clean'].astype(str))

q1_pad = pad_sequences(seq1, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
q2_pad = pad_sequences(seq2, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

y_deep = df['is_duplicate'].astype(int).values
vocab_size = min(MAX_NUM_WORDS, len(tokenizer.word_index) + 1)


In [ ]:
# Downloading small GloVe
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip -d glove

EMB_DIM = 100
emb_index = {}
with open('glove/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.rstrip().split(' ')
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        emb_index[word] = coefs

embedding_matrix = np.zeros((vocab_size, EMB_DIM))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    vec = emb_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

X1_train, X1_val, X2_train, X2_val, y_train, y_val = train_test_split(
    q1_pad, q2_pad, y_deep, test_size=0.2, random_state=42, stratify=y_deep
)

# Shared embedding + BiLSTM encoder
input_q = Input(shape=(MAX_SEQ_LEN,))
emb = Embedding(
    input_dim=vocab_size,
    output_dim=EMB_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQ_LEN,
    trainable=False
)(input_q)
x = Bidirectional(LSTM(128, return_sequences=True))(emb)
x = GlobalMaxPooling1D()(x)
x = Dropout(0.3)(x)
encoder = Model(inputs=input_q, outputs=x)

# Two branches share the encoder
q1_input = Input(shape=(MAX_SEQ_LEN,))
q2_input = Input(shape=(MAX_SEQ_LEN,))

q1_vec = encoder(q1_input)
q2_vec = encoder(q2_input)

# Combine representations
merged_mul = tf.keras.layers.Multiply()([q1_vec, q2_vec])
merged_abs = tf.keras.layers.Lambda(lambda z: tf.math.abs(z[0] - z[1]))([q1_vec, q2_vec])

merged = tf.keras.layers.Concatenate()([merged_mul, merged_abs])
merged = Dense(128, activation='relu')(merged)
merged = Dropout(0.3)(merged)
merged = Dense(64, activation='relu')(merged)
merged = Dropout(0.3)(merged)
output = Dense(1, activation='sigmoid')(merged)

model_bilstm = Model(inputs=[q1_input, q2_input], outputs=output)

model_bilstm.compile(
    loss='binary_crossentropy',
    optimizer=Adam(1e-3),
    metrics=['accuracy']
)

model_bilstm.summary()


In [ ]:
history = model_bilstm.fit(
    [X1_train, X2_train], y_train,
    validation_data=([X1_val, X2_val], y_val),
    epochs=5,
    batch_size=512,
    verbose=1
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

y_val_pred = (model_bilstm.predict([X1_val, X2_val])[:,0] >= 0.5).astype(int)
acc = accuracy_score(y_val, y_val_pred)
p, r, f1, _ = precision_recall_fscore_support(y_val, y_val_pred, average='binary')
print(f'BiLSTM: acc={acc:.3f}, precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}')


In [ ]:
def predict_duplicate_bilstm(q1, q2):
    s1 = tokenizer.texts_to_sequences([clean_text(str(q1))])
    s2 = tokenizer.texts_to_sequences([clean_text(str(q2))])
    p1 = pad_sequences(s1, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
    p2 = pad_sequences(s2, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
    prob = float(model_bilstm.predict([p1, p2])[0,0])
    label = int(prob >= 0.5)
    return label, prob


In [ ]:
q1 = "Do you feel like time is running faster now?"
q2 = "Why does time feel like it goes faster?"

label, prob = predict_duplicate_bilstm(q1, q2)
print("Duplicate?", bool(label), "Probability:", prob)
